In [1]:
import re
import torch
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForTokenClassification
from hazm import Normalizer

g:\python\NLP-Quera\NLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = r"G:\python\NLP-Quera\NER-HOME-ADS\model"  # مسیر پوشه حاوی config.json و model.safetensors

tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-base-uncased")
model = AutoModelForTokenClassification.from_pretrained(model_path)
model.to(device)
model.eval()

BertForTokenClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(100000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,

In [10]:
# ------------------------------------------------------------
# 2. توابع کمکی (Regex برای متراژ و تعداد اتاق)
# ------------------------------------------------------------
normalizer = Normalizer()

def extract_area(text):
    patterns = [
        r'(\d+(?:\.\d+)?)\s*متر',
        r'متراژ\s*(\d+(?:\.\d+)?)',
        r'زیربنا\s*(\d+(?:\.\d+)?)',
        r'(\d+(?:\.\d+)?)\s*متری',
        r'بنای\s*(\d+(?:\.\d+)?)\s*متر'
    ]
    for pat in patterns:
        m = re.search(pat, text)
        if m:
            num_str = m.group(1).translate(str.maketrans('۰۱۲۳۴۵۶۷۸۹', '0123456789'))
            try:
                return float(num_str)
            except:
                continue
    return None

def extract_rooms(text):
    rooms_map = {'یک':1, 'دو':2, 'سه':3, 'چهار':4, 'پنج':5}
    m = re.search(r'(\d+)\s*(?:خواب|اتاق)', text)
    if m:
        return int(m.group(1))
    for word, num in rooms_map.items():
        if re.search(rf'{word}\s*(?:خواب|اتاق)', text):
            return num
    return None

# ------------------------------------------------------------
# 3. تابع استخراج موجودیت‌های NER
# ------------------------------------------------------------
def extract_ner_entities(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=2)
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    labels = [model.config.id2label[p.item()] for p in predictions[0]]
    
    entities = {
        "has_elevator": None,
        "has_parking": None,
        "is_rebuilt": None
    }
    # اولویت: اگر هر Positive وجود داشت → True
    # اگر هیچ Positive نبود ولی Negative وجود داشت → False
    # در غیر این صورت None
    for token, label in zip(tokens, labels):
        if token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue
        if label.startswith("B-") or label.startswith("I-"):
            if "ELEVATOR_POS" in label:
                entities["has_elevator"] = True
            elif "ELEVATOR_NEG" in label:
                if entities["has_elevator"] is None:
                    entities["has_elevator"] = False
            elif "PARKING_POS" in label:
                entities["has_parking"] = True
            elif "PARKING_NEG" in label:
                if entities["has_parking"] is None:
                    entities["has_parking"] = False
            elif "REBUILT_POS" in label:
                entities["is_rebuilt"] = True
            elif "REBUILT_NEG" in label:
                if entities["is_rebuilt"] is None:
                    entities["is_rebuilt"] = False
    return entities

# ------------------------------------------------------------
# 4. تابع اصلی پردازش متن
# ------------------------------------------------------------
def process_text(text: str):
    # نرمال‌سازی
    normalized = normalizer.normalize(text)
    # استخراج با regex
    area = extract_area(normalized)
    rooms = extract_rooms(normalized)
    # استخراج با NER
    ner_results = extract_ner_entities(normalized)
    
    return {
        "building_size": area,
        "rooms_count": rooms,
        "has_elevator": ner_results["has_elevator"],
        "has_parking": ner_results["has_parking"],
        "is_rebuilt": ner_results["is_rebuilt"]
    }


In [11]:

# ------------------------------------------------------------
# 5. تعریف FastAPI
# ------------------------------------------------------------
app = FastAPI(title="Real Estate Info Extractor", version="1.0")

class TextInput(BaseModel):
    text: str

@app.post("/extract", response_model=dict)
async def extract(input: TextInput):
    if not input.text.strip():
        raise HTTPException(status_code=400, detail="متن ورودی خالی است")
    result = process_text(input.text)
    return result

@app.get("/health")
async def health():
    return {"status": "ok"}


In [13]:
import uvicorn

# برای اجرای مستقیم (uvicorn app:app --reload)
if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: Cannot run the event loop while another loop is running

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("HooshvareLab/bert-fa-base-uncased")
tokenizer.save_pretrained("./model")  # ذخیره در همان پوشه مدل

g:\python\NLP-Quera\NLP\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


('./model\\tokenizer_config.json',
 './model\\special_tokens_map.json',
 './model\\vocab.txt',
 './model\\added_tokens.json',
 './model\\tokenizer.json')